# 12.7 PyTorch Geometric in research: measured solubility regression

**Research question:** can a small graph model help estimate aqueous solubility for molecules outside its training scaffold groups? Solubility affects whether a compound can be prepared and tested at a desired concentration. Our model predicts a recorded laboratory endpoint; it does not replace a measurement at the required pH, temperature and formulation.

This is a complete **PyG implementation**: RDKit molecules → `Data` → graph `DataLoader` → `GINEConv` → graph readout → validation checkpoint → held-out assessment. We compare it with a training mean and descriptor ridge regression. Each stage has a short visible purpose.

## Before starting

Read [12.6](Chapter12_Part6.ipynb) for PyG tensors and batches, and [10.2](Chapter10_Part2.ipynb) for train/validation/test separation. **Regression** means predicting a number; a **baseline** is a simpler method the new model should be compared with; a **checkpoint** stores the chosen model parameters.

By the end you will train on mini-batches of complete molecules, preserve chemical groups, restore the best validation weights, interpret prediction errors in log units, and reload a complete inference model. The core path is the numbered workflow; the saved provenance and failure-case inspection are research habits worth practicing.

The notebook runs independently with local course data, CPU PyG, at most 400 molecules and 60 epochs. No prior output, GPU, large search or download is required. [Environment setup](Readme.md#set-up-python).

In [ ]:
import os
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
for key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
    os.environ[key] = '1'
from pathlib import Path
import hashlib
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
import torch
from torch import nn
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, GCNConv, GINEConv, global_mean_pool
from rdkit import Chem, rdBase, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Scaffolds import MurckoScaffold

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
SEED = 2026
torch.manual_seed(SEED)
OUT = Path('outputs/chapter12_part7')
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {'torch': str(torch.__version__), 'torch_geometric': torch_geometric.__version__,
            'rdkit': rdBase.rdkitVersion, 'numpy': np.__version__}
print(VERSIONS)
print('CPU only; no compiled graph extensions or dataset downloads.')

## 12.7.1 Know what a target value means

The local `Solubility.csv` contains measured Delaney/ESOL values:

$$y=\log_{10}\!\left(\frac{S}{1\ \mathrm{mol\,L^{-1}}}\right).$$

Thus $y=-3$ represents $1$ mmol/L; an error of one log unit represents a factor of ten for an individual concentration ratio. An average log error is not an error in mol/L. The CSV lacks measurement-condition detail, so we do not assign a universal temperature or pH. It corresponds to the **measured** source column, not the ESOL model's predictions. [Dataset provenance](datasets/README.md), [original Delaney paper](https://doi.org/10.1021/ci034243x).

We predeclare exactly the 400-row selection and group assignment used in 12.3. Repeated measurements remain within one partition. The architecture and optimizer here differ from 12.3, so a score difference cannot be attributed to choosing PyG alone.

In [ ]:
data_path = Path('datasets/Solubility.csv')
source_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert source_sha256 == '3fedd5ad80f9231bd331929ba0943a117d0d6ee3f75eda9a27fab9b4ab974ad5'
raw = pd.read_csv(data_path)
assert raw.notna().all().all() and np.isfinite(raw.solubility).all()
raw['source_row'] = np.arange(len(raw))
raw['smiles'] = raw.smiles.str.strip()
all_mols = [Chem.MolFromSmiles(s) for s in raw.smiles]
assert all(m is not None and m.GetNumAtoms() > 0 for m in all_mols)
raw['canonical'] = [Chem.MolToSmiles(m, isomericSmiles=True) for m in all_mols]
eligible = np.array([len(Chem.GetMolFrags(m)) == 1 and 1 <= m.GetNumHeavyAtoms() <= 60 for m in all_mols])
raw['selection_key'] = raw.smiles.map(lambda s: hashlib.sha256(s.encode()).hexdigest())
sample = raw.loc[eligible].sort_values(['selection_key','source_row']).head(400).copy().reset_index(drop=True)
molecules = [all_mols[i] for i in sample.source_row]
sample['group'] = [MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False) or 'ACYCLIC' for m in molecules]

def partition_for(group):
    fraction = int(hashlib.sha256(f'{SEED}|{group}'.encode()).hexdigest()[:8], 16)/2**32
    return 'train' if fraction < 0.70 else 'validation' if fraction < 0.85 else 'test'

sample['partition'] = sample.group.map(partition_for)
indices = {p: np.flatnonzero(sample.partition.eq(p)) for p in ['train','validation','test']}
train_id, val_id, test_id = [indices[p] for p in ['train','validation','test']]
assert sample.groupby('group').partition.nunique().max() == 1
assert sample.groupby('canonical').partition.nunique().max() == 1
assert all(len(i) >= 15 for i in indices.values())
display(sample.groupby('partition').agg(rows=('source_row','size'), groups=('group','nunique'),
    acyclic_rows=('group', lambda x: x.eq('ACYCLIC').sum())))
sample.drop(columns='selection_key').to_csv(OUT / 'split.csv', index=False)
print('Full source audit:', len(raw), 'parseable rows;', int(raw.canonical.duplicated().sum()), 'repeat extra rows.')

### Understand this particular split

A Murcko scaffold keeps ring systems and their connecting framework. Molecules with exactly the same scaffold stay together. All acyclic molecules share one group here. The resulting 245/19/136 train/validation/test counts are **not** a balanced random sample, and 111 test molecules are acyclic. The small validation set and dominant test group make this a limited teaching experiment, not an official benchmark or a representative deployment estimate.

**Predict:** if a model performs well on the most common held-out group but poorly on smaller groups, can the aggregate error hide that? Yes. We will retain group identities and inspect individual failures after freezing the model.

In [ ]:
composition = pd.crosstab(sample.partition, sample.group.eq('ACYCLIC')).reindex(['train','validation','test'], fill_value=0)
composition = composition.reindex(columns=[False,True], fill_value=0)
fig, ax = plt.subplots(figsize=(6.8, 3.4), layout='constrained')
ax.bar(composition.index, composition[False], label='Cyclic scaffold groups', color='#287f9d')
ax.bar(composition.index, composition[True], bottom=composition[False], label='One acyclic group', color='#d49132')
ax.set(ylabel='Recorded observations', title='Inspect the evaluation population before claiming generalization')
ax.legend(fontsize=8)
fig.savefig(OUT / 'split_composition.png', dpi=140)
plt.show()

## 12.7.2 Encode and batch graphs

We use the same 17 atom and 7 bond features defined in 12.6. The definitions are repeated so this notebook is independent. Element categories, fixed count divisors, charge and bond attributes are included; stereo, isotopes, spin, 3D geometry and measurement conditions are omitted. An `other` category prevents an indexing failure but does not establish reliable prediction for unfamiliar chemistry.

Only training labels determine the target mean $\mu_{\rm train}$ and standard deviation $s_{\rm train}$:

$$z=(y-\mu_{\rm train})/s_{\rm train},\qquad
\hat y=s_{\rm train}\hat z+\mu_{\rm train}.$$

This changes optimization scale, not the scientific target. A graph's standardized `y` has one entry; a mini-batch of 64 graphs has 64 target entries, regardless of how many atoms it contains.

In [ ]:
ELEMENTS = [5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]
NODE_NAMES = [f'element_{z}' for z in ELEMENTS] + ['element_other', 'degree/4',
              'attached_H/4', 'formal_charge/2', 'aromatic', 'in_ring']
EDGE_NAMES = ['single', 'double', 'triple', 'aromatic', 'other', 'conjugated', 'in_ring']
FEATURE_SCHEMA = {'elements': ELEMENTS, 'node_names': NODE_NAMES, 'edge_names': EDGE_NAMES,
    'hydrogens': 'RDKit default implicit H; attached counts on atoms',
    'components': 'connected only', 'stereochemistry': 'omitted', 'coordinates': 'omitted'}

def one_hot_other(value, choices):
    return [float(value == c) for c in choices] + [float(value not in choices)]

def mol_to_data(mol, target=None, source_row=None):
    if mol is None or mol.GetNumAtoms() == 0 or len(Chem.GetMolFrags(mol)) != 1:
        raise ValueError('Provide a nonempty connected molecule.')
    nodes = [one_hot_other(a.GetAtomicNum(), ELEMENTS) +
             [a.GetDegree()/4, a.GetTotalNumHs()/4, a.GetFormalCharge()/2,
              float(a.GetIsAromatic()), float(a.IsInRing())] for a in mol.GetAtoms()]
    edges, attributes = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        attr = one_hot_other(b.GetBondType(), BOND_TYPES) + [float(b.GetIsConjugated()), float(b.IsInRing())]
        edges.extend([(i, j), (j, i)])
        attributes.extend([attr, attr])
    data = Data(x=torch.tensor(nodes, dtype=torch.float32),
        edge_index=torch.tensor(edges, dtype=torch.long).reshape(-1, 2).T.contiguous(),
        edge_attr=torch.tensor(attributes, dtype=torch.float32).reshape(-1, len(EDGE_NAMES)),
        num_nodes=mol.GetNumAtoms())
    if target is not None:
        data.y = torch.tensor([target], dtype=torch.float32)
    if source_row is not None:
        data.source_row = torch.tensor([source_row], dtype=torch.long)
    data.validate(raise_on_error=True)
    return data

In [ ]:
y = sample.solubility.to_numpy(dtype=float)
y_mean, y_std = float(y[train_id].mean()), float(y[train_id].std())
assert y_std > 0
graphs = [mol_to_data(m, target=(y[i]-y_mean)/y_std, source_row=int(sample.iloc[i].source_row))
          for i, m in enumerate(molecules)]
datasets = {p: [graphs[i] for i in ids] for p, ids in indices.items()}
train_loader = DataLoader(datasets['train'], batch_size=64, shuffle=True, num_workers=0,
                         generator=torch.Generator().manual_seed(SEED))
full_batches = {p: Batch.from_data_list(items) for p, items in datasets.items()}
encoding_audit = {'other_element_atoms': sum(a.GetAtomicNum() not in ELEMENTS for m in molecules for a in m.GetAtoms()),
    'specified_chiral_atoms': sum(a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED for m in molecules for a in m.GetAtoms()),
    'isotope_atoms': sum(a.GetIsotope() != 0 for m in molecules for a in m.GetAtoms()),
    'radical_atoms': sum(a.GetNumRadicalElectrons() != 0 for m in molecules for a in m.GetAtoms())}
print('Encoding audit:', encoding_audit)
print('Training graphs:', len(datasets['train']), '| first graph:', datasets['train'][0])
assert full_batches['train'].y.shape == (len(train_id),)

## 12.7.3 Build a small bond-aware model with `GINEConv`

Each GINE layer combines an atom's state with a sum of neighbor states transformed by bond features. The layer's MLP learns how to use that combination. We apply two such layers, add a residual connection after each, average atom embeddings per graph, append $\log(1+N)$ to retain an explicit size signal, and predict one unrestricted scalar. Log solubility can be negative, so the regression head has no sigmoid.

The hidden width is 24. Mean pooling plus a size feature does not enforce an additive physical law; log solubility is not an extensive energy. This is a small teaching architecture using a real [PyG GINE operator](https://pytorch-geometric.readthedocs.io/en/2.8.0/generated/torch_geometric.nn.conv.GINEConv.html), not a reproduction of a published benchmark model.

In [ ]:
class PropertyGINE(nn.Module):
    def __init__(self, node_dim=17, edge_dim=7, hidden=24, layers=2):
        super().__init__()
        self.encoder = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList([GINEConv(
            nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)),
            edge_dim=edge_dim, train_eps=False) for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden+1, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x, edge_index, edge_attr, batch):
        h = torch.relu(self.encoder(x))
        for conv in self.convs:
            h = h + torch.relu(conv(h, edge_index, edge_attr))
        pooled = global_mean_pool(h, batch)
        counts = torch.bincount(batch, minlength=len(pooled)).to(h.dtype).unsqueeze(1)
        return self.head(torch.cat([pooled, torch.log1p(counts)], dim=1)).squeeze(1)

def forward_batch(model, data):
    return model(data.x, data.edge_index, data.edge_attr, data.batch)

ARCHITECTURE = {'node_dim': len(NODE_NAMES), 'edge_dim': len(EDGE_NAMES), 'hidden': 24, 'layers': 2}

In [ ]:
torch.manual_seed(SEED)
model = PropertyGINE(**ARCHITECTURE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
loss_function = nn.MSELoss()
MAX_EPOCHS, PATIENCE, MIN_DELTA = 60, 10, 1e-4
assert forward_batch(model, full_batches['validation']).shape == full_batches['validation'].y.shape
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 12.7.4 Train, monitor validation, restore the checkpoint

An epoch visits every training graph once in shuffled mini-batches. The model sees training labels in gradient calculations; validation labels select the checkpoint; test labels do neither. After each epoch we compute training and validation errors with the same frozen post-update parameters, so the curves are comparable.

We predeclare 60 epochs, patience 10, learning rate 0.005 and weight decay 0.0001. The lowest validation loss determines the saved weights; a separate tolerance determines when patience runs out. We do not modify these choices to improve the final test score. Small validation groups make model selection noisy even with correct code.

In [ ]:
best_loss, patience_reference = float('inf'), float('inf')
best_state, best_epoch, stale = None, None, 0
history_rows = []
start = time.perf_counter()
for epoch in range(1, MAX_EPOCHS+1):
    model.train()
    for mini_batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = loss_function(forward_batch(model, mini_batch), mini_batch.y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.inference_mode():
        train_loss = loss_function(forward_batch(model, full_batches['train']), full_batches['train'].y).item()
        val_loss = loss_function(forward_batch(model, full_batches['validation']), full_batches['validation'].y).item()
    assert np.isfinite([train_loss, val_loss]).all()
    history_rows.append({'epoch': epoch, 'train_RMSE_logS': np.sqrt(train_loss)*y_std,
                         'validation_RMSE_logS': np.sqrt(val_loss)*y_std})
    if val_loss < best_loss:
        best_loss, best_epoch = val_loss, epoch
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    if val_loss < patience_reference-MIN_DELTA:
        patience_reference, stale = val_loss, 0
    else:
        stale += 1
    if stale >= PATIENCE:
        break
model.load_state_dict(best_state)
model.eval()
training_seconds = time.perf_counter()-start
with torch.inference_mode():
    restored_loss = loss_function(forward_batch(model, full_batches['validation']), full_batches['validation'].y).item()
assert np.isclose(restored_loss, best_loss, rtol=1e-6)
history = pd.DataFrame(history_rows)
history.to_csv(OUT / 'learning_history.csv', index=False)
print(f'{epoch} epochs in {training_seconds:.2f} s; restored epoch {best_epoch}.')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), layout='constrained')
ax.plot(history.epoch, history.train_RMSE_logS, label='Training')
ax.plot(history.epoch, history.validation_RMSE_logS, label='Validation')
ax.axvline(best_epoch, color='black', linestyle=':', label=f'Restored epoch {best_epoch}')
ax.set(xlabel='Epoch', ylabel='RMSE (log solubility units)', title='Validation chooses the checkpoint')
ax.legend(fontsize=8)
fig.savefig(OUT / 'learning_curves.png', dpi=140)
plt.show()

## 12.7.5 Compare frozen predictions with useful baselines

The training-mean baseline ignores structure. A ridge model uses nine chemically motivated RDKit descriptors with a scaler fitted on training rows only. Its regularization parameter is fixed at 1. Baseline and GNN recipes use identical training records, but are not exhaustive searches for the best model in each family.

**Predict:** should a larger or newer model always win? No. Useful descriptors can be competitive when data are sparse. We report MAE, RMSE and $R^2$ in the held-out population and inspect the scatter plot. No accuracy threshold or model-ranking claim is imposed.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DESCRIPTOR_NAMES = ['MolWt','NumHeteroatoms','RingCount','NumHAcceptors','NumHDonors',
                    'FractionCSP3','TPSA','MolLogP','MolMR']
functions = dict(Descriptors.descList)
X_desc = np.array([[functions[name](m) for name in DESCRIPTOR_NAMES] for m in molecules])
assert np.isfinite(X_desc).all()
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_desc[train_id], y[train_id])
with torch.inference_mode():
    pyg_prediction = forward_batch(model, full_batches['test']).numpy()*y_std+y_mean
predictions = {'Training mean': np.full(len(test_id), y_mean),
               'Descriptor ridge': ridge.predict(X_desc[test_id]), 'PyG GINE': pyg_prediction}
metrics = pd.DataFrame({name: {'MAE': mean_absolute_error(y[test_id], pred),
    'RMSE': np.sqrt(mean_squared_error(y[test_id], pred)), 'R2': r2_score(y[test_id], pred)}
    for name, pred in predictions.items()}).T
display(metrics.round(3))
metrics.to_csv(OUT / 'test_metrics.csv')
test_table = sample.iloc[test_id][['source_row','smiles','canonical','group','solubility']].copy()
for name, pred in predictions.items(): test_table[name] = pred
test_table['absolute_error'] = abs(pyg_prediction-y[test_id])
test_table.to_csv(OUT / 'test_predictions.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(9.3, 4), layout='constrained')
limits = [min(y[test_id].min(), *(p.min() for p in predictions.values()))-0.2,
          max(y[test_id].max(), *(p.max() for p in predictions.values()))+0.2]
for ax, name in zip(axes, ['Descriptor ridge', 'PyG GINE']):
    ax.scatter(y[test_id], predictions[name], s=20, alpha=0.65)
    ax.plot(limits, limits, 'k--', lw=1)
    ax.set(xlabel='Measured log solubility', ylabel='Predicted log solubility',
           xlim=limits, ylim=limits, title=name)
fig.savefig(OUT / 'test_parity.png', dpi=140)
plt.show()

## 12.7.6 Research follow-up: inspect failures before adding layers

An aggregate metric does not show which compounds failed. We inspect the four largest absolute test residuals after the protocol is frozen. This is a **post-assessment audit**, not a second tuning set. A future model change needs a newly specified evaluation procedure.

For each failure, ask whether the recorded chemical state, experimental conditions, training coverage, repeated measurements, or omitted representation details might matter. A picture suggests questions; it does not identify the cause of an error. Do not simply remove difficult observations because they lower performance.

In [ ]:
worst = test_table.sort_values('absolute_error', ascending=False).head(4)
display(worst[['source_row','solubility','PyG GINE','absolute_error','group']])
case_mols = [Chem.MolFromSmiles(s) for s in worst.smiles]
legends = [f'Row {int(r.source_row)} | measured {r.solubility:.2f} | model {r["PyG GINE"]:.2f}'
           for _, r in worst.iterrows()]
drawer = rdMolDraw2D.MolDraw2DCairo(1080, 300, 270, 300)
drawer.DrawMolecules(case_mols, legends=legends)
drawer.FinishDrawing()
png = drawer.GetDrawingText()
(OUT / 'failure_cases.png').write_bytes(png)
display(Image(data=png))

## 12.7.7 Save a usable model, then verify reloading

The checkpoint includes feature definitions, architecture, target scale, dataset hash, split record IDs, versions and selected weights. It supports inference; it does not store optimizer state for exact training resumption. The original CSV and split file remain part of the experiment record.

We reconstruct a model, repeat predictions for two validation molecules, and compare separate, batched and atom-renumbered inference. Successful serialization means predictions are reproduced, not that they are scientifically correct for arbitrary new inputs.

In [ ]:
checkpoint = {'state_dict': model.state_dict(), 'architecture': ARCHITECTURE,
    'feature_schema': FEATURE_SCHEMA, 'target_mean': y_mean, 'target_std': y_std,
    'target': 'measured log10(S / (1 mol/L))', 'source_sha256': source_sha256,
    'versions': VERSIONS, 'best_epoch': best_epoch, 'seed': SEED,
    'rows_by_partition': {p: sample.iloc[ids].source_row.tolist() for p, ids in indices.items()}}
torch.save(checkpoint, OUT / 'pyg_solubility.pt')
saved = torch.load(OUT / 'pyg_solubility.pt', map_location='cpu', weights_only=True)
assert saved['feature_schema'] == FEATURE_SCHEMA
restored = PropertyGINE(**saved['architecture']).eval()
restored.load_state_dict(saved['state_dict'])
probe = Batch.from_data_list([graphs[i] for i in val_id[:2]])
renumbered = mol_to_data(Chem.RenumberAtoms(molecules[val_id[0]],
    list(reversed(range(molecules[val_id[0]].GetNumAtoms())))))
with torch.inference_mode():
    original_values = forward_batch(model, probe)
    restored_values = forward_batch(restored, probe)
    separate_values = torch.cat([forward_batch(restored, Batch.from_data_list([graphs[i]])) for i in val_id[:2]])
    permuted_value = forward_batch(restored, Batch.from_data_list([renumbered]))[0]
torch.testing.assert_close(original_values, restored_values)
torch.testing.assert_close(restored_values, separate_values, atol=2e-6, rtol=2e-6)
torch.testing.assert_close(restored_values[0], permuted_value, atol=2e-6, rtol=2e-6)
record = {'scope': 'Fixed small measured-solubility PyG experiment; not an official benchmark',
    'source_sha256': source_sha256, 'versions': VERSIONS, 'feature_schema': FEATURE_SCHEMA,
    'architecture': ARCHITECTURE, 'encoding_audit': encoding_audit,
    'split_counts': {p: len(ids) for p,ids in indices.items()}, 'seed': SEED,
    'training': {'max_epochs': MAX_EPOCHS, 'epochs_run': epoch, 'best_epoch': best_epoch,
        'patience': PATIENCE, 'min_delta': MIN_DELTA, 'batch_size': 64, 'learning_rate': 0.005,
        'optimizer': 'Adam', 'weight_decay': 1e-4, 'seconds': training_seconds},
    'test_metrics': metrics.to_dict(orient='index')}
(OUT / 'experiment.json').write_text(json.dumps(record,indent=2)+'\n', encoding='utf-8')
print('Reloading, batching and atom-permutation checks passed; artifacts saved to', OUT)

## Exercises and suggested answers

1. Why does the same `DataLoader` batch size correspond to different numbers of atoms across batches?
2. Which labels influence weights, target scaling, checkpoint selection and final assessment?
3. Why is the test group's composition scientifically important even though all tensor checks pass?
4. A descriptor baseline beats this GNN. What would be a defensible next experiment?
5. Which extra metadata would a collaborator need before trusting a prediction for a new ionizable compound?

<details><summary>Answers</summary>

1. Molecules vary in size. A batch contains a number of whole graphs, each with one molecular target.
2. Training labels fit weights and target statistics; validation labels choose the checkpoint; test labels assess the frozen procedure.
3. A metric averages over the sampled population. One large acyclic group dominates much of this test set, while the validation set is small. The result may not represent the future compounds of interest.
4. Predeclare a new hypothesis, such as richer features or a matched hyperparameter budget, use appropriate group-based selection, and assess it without repeatedly tuning against the same final test labels. First inspect data quality and learning curves.
5. Chemical state, assay pH/temperature/solvent, measurement protocol, similar training examples, feature support, and relevant validation evidence. A valid SMILES alone does not provide these.

</details>

**References:** [PyG graph mini-batches](https://pytorch-geometric.readthedocs.io/en/2.8.0/advanced/batching.html), [GINEConv](https://pytorch-geometric.readthedocs.io/en/2.8.0/generated/torch_geometric.nn.conv.GINEConv.html), [MoleculeNet](https://doi.org/10.1039/C7SC02664A), [data and representation evaluation](https://www.nature.com/articles/s41467-023-41948-6).

[Next: 12.8 — graph classification and a real PyG explainer](Chapter12_Part8.ipynb) · [Course guide](docs/course-guide.md)